# GéoMarketing IDF — J7 : Profil détaillé de la clientèle potentielle

## 07c - Scolarisation et diplômes

In [1]:
#Importation des librairies
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

from pathlib import Path
from zipfile import ZipFile
import re
import shutil
import unicodedata

import numpy as np
import pandas as pd
import openpyxl

from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 150)

print("Pandas :", pd.__version__)
print("NumPy :", np.__version__)
print("Openpyxl :", openpyxl.__version__)
print("Importations réussies ✅")

Pandas : 2.2.2
NumPy : 1.26.4
Openpyxl : 3.1.5
Importations réussies ✅


In [2]:
#Dossiers
RACINE = Path(
    r"C:\Users\almou\OneDrive\GeoMarketing_IDF"
)

DOSSIER_RAW = (
    RACINE
    / "data"
    / "raw"
    / "insee"
    / "rp2023"
)

DOSSIER_INTERIM = (
    RACINE
    / "data"
    / "interim"
)

DOSSIER_PROCESSED = (
    RACINE
    / "data"
    / "processed"
)

for dossier in [
    DOSSIER_RAW,
    DOSSIER_INTERIM,
    DOSSIER_PROCESSED,
]:
    dossier.mkdir(
        parents=True,
        exist_ok=True,
    )

print("Racine :", RACINE)
print("Raw :", DOSSIER_RAW)
print("Interim :", DOSSIER_INTERIM)
print("Processed :", DOSSIER_PROCESSED)

assert RACINE.exists(), "Le dossier du projet n'existe pas."

Racine : C:\Users\almou\OneDrive\GeoMarketing_IDF
Raw : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\raw\insee\rp2023
Interim : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\interim
Processed : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed


In [3]:
def normaliser_nom_colonne(nom):
    nom = str(nom).strip().upper()

    nom = unicodedata.normalize(
        "NFKD",
        nom,
    )

    nom = "".join(
        caractere
        for caractere in nom
        if not unicodedata.combining(caractere)
    )

    nom = re.sub(
        r"[^A-Z0-9]+",
        "_",
        nom,
    )

    return nom.strip("_")


def normaliser_code_commune(serie):
    return (
        serie.astype("string")
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True,
        )
        .str.upper()
        .str.zfill(5)
    )


def pourcentage(numerateur, denominateur):
    numerateur = pd.to_numeric(
        numerateur,
        errors="coerce",
    )

    denominateur = pd.to_numeric(
        denominateur,
        errors="coerce",
    )

    return (
        numerateur
        .div(
            denominateur.where(
                denominateur.ne(0)
            )
        )
        .mul(100)
    )


def verifier_classeur_xlsx(fichier):
    if not fichier.exists():
        raise FileNotFoundError(
            f"Classeur introuvable : {fichier}"
        )

    with open(fichier, "rb") as flux:
        signature = flux.read(4)

    if signature != b"PK\x03\x04":
        raise ValueError(
            f"{fichier.name} n'est pas un véritable fichier XLSX."
        )

    with ZipFile(fichier) as archive:
        noms = set(archive.namelist())

        if "xl/workbook.xml" not in noms:
            raise ValueError(
                f"{fichier.name} ne contient pas de classeur Excel valide."
            )

    print(
        f"Classeur valide : {fichier.name} "
        f"({fichier.stat().st_size / 1_000_000:.2f} Mo)"
    )


def enregistrer_csv(table, fichier):
    table.to_csv(
        fichier,
        index=False,
        sep=",",
        encoding="utf-8-sig",
    )

    print(
        "Fichier CSV créé :",
        fichier,
    )


In [9]:
noms_profils_j6 = [
    "profil_communes_idf_j6.xlsx",
    "profil_communes_idf_j6.csv",
    "profil_communes_idf.csv",
]

FICHIER_PROFIL_J6 = None

for nom in noms_profils_j6:
    candidat = DOSSIER_PROCESSED / nom

    if candidat.exists():
        FICHIER_PROFIL_J6 = candidat
        break

if FICHIER_PROFIL_J6 is None:
    raise FileNotFoundError(
        "Le profil J6 est introuvable dans data/processed."
    )

if FICHIER_PROFIL_J6.suffix.lower() == ".xlsx":
    profil_j6 = pd.read_excel(
        FICHIER_PROFIL_J6,
        sheet_name=0,
        engine="openpyxl",
    )

else:
    profil_j6 = pd.read_csv(
        FICHIER_PROFIL_J6,
        sep=None,
        engine="python",
        encoding="utf-8-sig",
    )

profil_j6.columns = [
    normaliser_nom_colonne(colonne)
    for colonne in profil_j6.columns
]


profil_j6.columns = [
    normaliser_nom_colonne(colonne)
    for colonne in profil_j6.columns
]

if "CODGEO" not in profil_j6.columns:
    candidats_code = [
        "DEPCOM",
        "CODE_COMMUNE",
        "COM",
    ]

    colonne_code = next(
        (
            colonne
            for colonne in candidats_code
            if colonne in profil_j6.columns
        ),
        None,
    )

    if colonne_code is None:
        raise ValueError(
            "Aucune colonne de code communal dans le profil J6."
        )

    profil_j6 = profil_j6.rename(
        columns={
            colonne_code: "CODGEO"
        }
    )

profil_j6["CODGEO"] = normaliser_code_commune(
    profil_j6["CODGEO"]
)

assert profil_j6["CODGEO"].notna().all()
assert profil_j6["CODGEO"].is_unique
assert profil_j6["CODGEO"].str.fullmatch(
    r"\d{5}"
).all()

codes_profil = set(
    profil_j6["CODGEO"]
)

print("Profil J6 :", FICHIER_PROFIL_J6)
print("Nombre de communes :", len(profil_j6))
print("Profil J6 chargé ✅")
    

Profil J6 : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j6.csv
Nombre de communes : 1266
Profil J6 chargé ✅


In [10]:
#Récupérer le fichier source
FICHIER_DIPLOMES = (
    DOSSIER_RAW
    / "base_cc_diplomes_formation_2023.xlsx"
)

verifier_classeur_xlsx(
    FICHIER_DIPLOMES
)

classeur_diplomes = pd.ExcelFile(
    FICHIER_DIPLOMES,
    engine="openpyxl",
)

print(
    "Onglets :",
    classeur_diplomes.sheet_names,
)

assert "COM_2023" in classeur_diplomes.sheet_names

Classeur valide : base_cc_diplomes_formation_2023.xlsx (44.15 Mo)
Onglets : ['Métadonnées', 'COM_2023', 'ARM_2023', 'COM_2017', 'ARM_2017', 'COM_2012', 'ARM_2012', 'Documentation']


In [11]:
#Définir les colonnes de scolarisation
colonnes_diplomes = {
    "Code géographique": "CODGEO",
    "Libellé géographique": "LIBGEO",
}

tranches_scolarite = {
    "2_5": "2-5 ans",
    "6_10": "6-10 ans",
    "11_14": "11-14 ans",
    "15_17": "15-17 ans",
    "18_24": "18-24 ans",
    "25_29": "25-29 ans",
    "30_PLUS": "30 ans ou plus",
}

for code_tranche, libelle_tranche in tranches_scolarite.items():
    colonnes_diplomes[
        f"Pop {libelle_tranche} (princ)"
    ] = (
        f"POP_REFERENCE_SCOLARITE_{code_tranche}"
    )

    colonnes_diplomes[
        f"Pop scolarisée {libelle_tranche} (princ)"
    ] = (
        f"POP_SCOLARISEE_{code_tranche}"
    )

In [12]:
#Définir les colonnes de diplômes
colonnes_diplomes[
    "Pop 15 ans ou plus non scolarisée (princ)"
] = "POP_NON_SCOL_15P"

colonnes_diplomes[
    "Hommes 15 ans ou plus non scolarisés (princ)"
] = "HOMMES_NON_SCOL_15P"

colonnes_diplomes[
    "Femmes 15 ans ou plus non scolarisées (princ)"
] = "FEMMES_NON_SCOL_15P"


modalites_diplomes = {
    "SANS_DIPLOME_CEP": "Sans diplôme ou CEP",
    "BEPC_DNB": "BEPC, brevet des collèges, DNB",
    "CAP_BEP": "CAP-BEP ou équiv.",
    "BAC": "Bac, brevet pro. ou équiv.",
    "BAC2": "Enseignement sup de niveau bac + 2",
    "BAC3_4": "Enseignement sup de niveau bac + 3 ou 4",
    "BAC5_PLUS": "Enseignement sup de niveau bac + 5 ou plus",
}

for code_diplome, libelle_diplome in modalites_diplomes.items():
    colonnes_diplomes[
        "Pop 15 ans ou plus non scol. "
        f"{libelle_diplome} (princ)"
    ] = (
        f"POP_NON_SCOL_{code_diplome}"
    )

    colonnes_diplomes[
        "Hommes 15 ans ou plus non scol. "
        f"{libelle_diplome} (princ)"
    ] = (
        f"HOMMES_NON_SCOL_{code_diplome}"
    )

    colonnes_diplomes[
        "Femmes 15 ans ou plus non scol. "
        f"{libelle_diplome} (princ)"
    ] = (
        f"FEMMES_NON_SCOL_{code_diplome}"
    )

print(
    "Colonnes demandées :",
    len(colonnes_diplomes),
)

Colonnes demandées : 40


In [13]:
#Charger le classeur
entete_diplomes = pd.read_excel(
    FICHIER_DIPLOMES,
    sheet_name="COM_2023",
    nrows=0,
    engine="openpyxl",
)

colonnes_absentes = (
    set(colonnes_diplomes)
    - set(entete_diplomes.columns)
)

if colonnes_absentes:
    raise ValueError(
        "Colonnes absentes du classeur diplômes : "
        f"{sorted(colonnes_absentes)}"
    )

diplomes_source = pd.read_excel(
    FICHIER_DIPLOMES,
    sheet_name="COM_2023",
    usecols=list(colonnes_diplomes),
    dtype={
        "Code géographique": "string",
    },
    engine="openpyxl",
)

diplomes_source = diplomes_source.rename(
    columns=colonnes_diplomes
)

diplomes_source["CODGEO"] = (
    normaliser_code_commune(
        diplomes_source["CODGEO"]
    )
)

display(
    diplomes_source.head()
)

,CODGEO,LIBGEO,POP_REFERENCE_SCOLARITE_2_5,POP_REFERENCE_SCOLARITE_6_10,POP_REFERENCE_SCOLARITE_11_14,POP_REFERENCE_SCOLARITE_15_17,POP_REFERENCE_SCOLARITE_18_24,POP_REFERENCE_SCOLARITE_25_29,POP_REFERENCE_SCOLARITE_30_PLUS,POP_SCOLARISEE_2_5,POP_SCOLARISEE_6_10,POP_SCOLARISEE_11_14,POP_SCOLARISEE_15_17,POP_SCOLARISEE_18_24,POP_SCOLARISEE_25_29,POP_SCOLARISEE_30_PLUS,POP_NON_SCOL_15P,POP_NON_SCOL_SANS_DIPLOME_CEP,POP_NON_SCOL_BEPC_DNB,POP_NON_SCOL_CAP_BEP,POP_NON_SCOL_BAC,POP_NON_SCOL_BAC2,POP_NON_SCOL_BAC3_4,POP_NON_SCOL_BAC5_PLUS,HOMMES_NON_SCOL_15P,HOMMES_NON_SCOL_SANS_DIPLOME_CEP,HOMMES_NON_SCOL_BEPC_DNB,HOMMES_NON_SCOL_CAP_BEP,HOMMES_NON_SCOL_BAC,HOMMES_NON_SCOL_BAC2,HOMMES_NON_SCOL_BAC3_4,HOMMES_NON_SCOL_BAC5_PLUS,FEMMES_NON_SCOL_15P,FEMMES_NON_SCOL_SANS_DIPLOME_CEP,FEMMES_NON_SCOL_BEPC_DNB,FEMMES_NON_SCOL_CAP_BEP,FEMMES_NON_SCOL_BAC,FEMMES_NON_SCOL_BAC2,FEMMES_NON_SCOL_BAC3_4,FEMMES_NON_SCOL_BAC5_PLUS
0,01001,L'Abergement-Clémenciat,33.74986,55.55884,44.66127,35.82970,29.97268,37.09322,596.29927,25.77755,52.57719,42.68762,33.84191,9.95305,1.00096,0.00000,654.39895,101.62019,34.19085,194.09548,131.26465,89.13041,63.14884,40.94852,322.14433,44.19305,17.06711,111.61519,64.11328,42.10521,23.08725,19.96325,332.25462,57.42714,17.12375,82.48029,67.15137,47.02520,40.06160,20.98528
1,01002,L'Abergement-de-Varey,13.68887,18.58177,19.62103,13.66559,10.69066,6.83383,183.00180,11.72434,16.62984,19.62103,12.66427,6.77196,0.00000,1.97712,192.77852,18.91561,6.04664,41.99418,41.55558,29.62271,28.72248,25.92132,102.36354,10.94048,1.01423,25.99818,23.62008,16.84571,11.90227,12.04259,90.41498,7.97513,5.03241,15.99600,17.93550,12.77700,16.82020,13.87873
2,01004,Ambérieu-en-Bugey,872.11821,991.68589,796.77941,617.89119,1475.39486,1127.14235,9608.23936,607.68370,971.67549,787.20868,580.95995,534.18246,30.70796,68.16479,11614.65261,2302.49600,710.17845,2772.55107,2348.02294,1359.57799,1018.17765,1103.64851,5547.09672,916.82632,279.84196,1559.17535,1125.37497,655.00705,410.74163,600.12944,6067.55589,1385.66968,430.33649,1213.37572,1222.64797,704.57094,607.43602,503.51907
3,01005,Ambérieux-en-Dombes,92.58332,122.67862,96.72355,60.41435,106.62758,120.65357,1266.27835,63.78518,116.03889,92.80916,56.56754,42.65087,2.13596,6.56910,1446.05038,260.50989,80.16507,421.99063,276.33575,201.73274,111.51667,93.79963,722.46875,120.47629,39.44734,247.47858,122.63195,95.10782,44.86963,52.45715,723.58164,140.03360,40.71773,174.51206,153.70380,106.62493,66.64703,41.34248
4,01006,Ambléon,4.03509,4.03509,0.00000,2.01754,6.05263,6.05263,90.78947,3.02632,2.01754,0.00000,1.00877,1.00877,0.00000,1.00877,101.88596,20.17544,8.07018,28.24561,16.14035,12.10526,11.09649,6.05263,55.48246,11.09649,1.00877,21.18421,7.06140,6.05263,6.05263,3.02632,46.40351,9.07895,7.06140,7.06140,9.07895,6.05263,5.04386,3.02632


In [14]:
#Filtrer les communes d'Ile-de-France
colonnes_numeriques_diplomes = [
    colonne
    for colonne in diplomes_source.columns
    if colonne not in [
        "CODGEO",
        "LIBGEO",
    ]
]

diplomes_source[
    colonnes_numeriques_diplomes
] = diplomes_source[
    colonnes_numeriques_diplomes
].apply(
    pd.to_numeric,
    errors="coerce",
)

codes_diplomes = set(
    diplomes_source["CODGEO"]
)

codes_absents_diplomes = sorted(
    codes_profil - codes_diplomes
)

if codes_absents_diplomes:
    raise ValueError(
        "Codes du profil absents du classeur diplômes : "
        f"{codes_absents_diplomes}"
    )

diplomes_idf_source = diplomes_source[
    diplomes_source["CODGEO"].isin(
        codes_profil
    )
].copy()

diplomes_idf_source = (
    diplomes_idf_source
    .sort_values("CODGEO")
    .reset_index(drop=True)
)

assert len(diplomes_idf_source) == len(profil_j6)
assert diplomes_idf_source["CODGEO"].is_unique

print(
    "Communes IDF :",
    len(diplomes_idf_source),
)

Communes IDF : 1266


In [15]:
indicateurs_7c = diplomes_idf_source.copy()

for code_tranche in tranches_scolarite:
    indicateurs_7c[
        f"TAUX_SCOLARISATION_{code_tranche}_PCT"
    ] = pourcentage(
        indicateurs_7c[
            f"POP_SCOLARISEE_{code_tranche}"
        ],
        indicateurs_7c[
            f"POP_REFERENCE_SCOLARITE_{code_tranche}"
        ],
    )

display(
    indicateurs_7c[
        [
            "CODGEO",
            "LIBGEO",
            "POP_SCOLARISEE_18_24",
            "TAUX_SCOLARISATION_18_24_PCT",
            "POP_SCOLARISEE_25_29",
            "TAUX_SCOLARISATION_25_29_PCT",
        ]
    ].head()
)

,CODGEO,LIBGEO,POP_SCOLARISEE_18_24,TAUX_SCOLARISATION_18_24_PCT,POP_SCOLARISEE_25_29,TAUX_SCOLARISATION_25_29_PCT
0,75056,Paris,171598.23785,72.616658,36454.70768,16.683169
1,77001,Achères-la-Forêt,38.12267,57.233014,0.00000,0.000000
2,77002,Amillis,22.36168,40.260137,1.00021,2.589905
3,77003,Amponville,7.88436,44.353012,1.97621,13.235112
4,77004,Andrezel,4.20925,33.688095,2.03827,14.284383


In [16]:
#Regrouper les niveaux de diplôme

groupes_diplomes = {
    "POP": {
        "total": "POP_NON_SCOL_15P",
        "prefixe": "POP_NON_SCOL_",
    },
    "HOMMES": {
        "total": "HOMMES_NON_SCOL_15P",
        "prefixe": "HOMMES_NON_SCOL_",
    },
    "FEMMES": {
        "total": "FEMMES_NON_SCOL_15P",
        "prefixe": "FEMMES_NON_SCOL_",
    },
}

for groupe, informations in groupes_diplomes.items():
    total = informations["total"]
    prefixe = informations["prefixe"]

    indicateurs_7c[
        f"{groupe}_NON_SCOL_DIPLOME_SUP"
    ] = (
        indicateurs_7c[
            f"{prefixe}BAC2"
        ]
        + indicateurs_7c[
            f"{prefixe}BAC3_4"
        ]
        + indicateurs_7c[
            f"{prefixe}BAC5_PLUS"
        ]
    )

    indicateurs_7c[
        f"{groupe}_NON_SCOL_BAC_OU_PLUS"
    ] = (
        indicateurs_7c[
            f"{prefixe}BAC"
        ]
        + indicateurs_7c[
            f"{groupe}_NON_SCOL_DIPLOME_SUP"
        ]
    )

    indicateurs_7c[
        f"PART_{groupe}_SANS_DIPLOME_CEP_PCT"
    ] = pourcentage(
        indicateurs_7c[
            f"{prefixe}SANS_DIPLOME_CEP"
        ],
        indicateurs_7c[total],
    )

    indicateurs_7c[
        f"PART_{groupe}_BAC_OU_PLUS_PCT"
    ] = pourcentage(
        indicateurs_7c[
            f"{groupe}_NON_SCOL_BAC_OU_PLUS"
        ],
        indicateurs_7c[total],
    )

    indicateurs_7c[
        f"PART_{groupe}_DIPLOME_SUP_PCT"
    ] = pourcentage(
        indicateurs_7c[
            f"{groupe}_NON_SCOL_DIPLOME_SUP"
        ],
        indicateurs_7c[total],
    )

    indicateurs_7c[
        f"PART_{groupe}_BAC5_PLUS_PCT"
    ] = pourcentage(
        indicateurs_7c[
            f"{prefixe}BAC5_PLUS"
        ],
        indicateurs_7c[total],
    )

In [17]:
# Comparaison hommes-femmes
groupes_diplomes = {
    "POP": {
        "total": "POP_NON_SCOL_15P",
        "prefixe": "POP_NON_SCOL_",
    },
    "HOMMES": {
        "total": "HOMMES_NON_SCOL_15P",
        "prefixe": "HOMMES_NON_SCOL_",
    },
    "FEMMES": {
        "total": "FEMMES_NON_SCOL_15P",
        "prefixe": "FEMMES_NON_SCOL_",
    },
}

for groupe, informations in groupes_diplomes.items():
    total = informations["total"]
    prefixe = informations["prefixe"]

    indicateurs_7c[
        f"{groupe}_NON_SCOL_DIPLOME_SUP"
    ] = (
        indicateurs_7c[
            f"{prefixe}BAC2"
        ]
        + indicateurs_7c[
            f"{prefixe}BAC3_4"
        ]
        + indicateurs_7c[
            f"{prefixe}BAC5_PLUS"
        ]
    )

    indicateurs_7c[
        f"{groupe}_NON_SCOL_BAC_OU_PLUS"
    ] = (
        indicateurs_7c[
            f"{prefixe}BAC"
        ]
        + indicateurs_7c[
            f"{groupe}_NON_SCOL_DIPLOME_SUP"
        ]
    )

    indicateurs_7c[
        f"PART_{groupe}_SANS_DIPLOME_CEP_PCT"
    ] = pourcentage(
        indicateurs_7c[
            f"{prefixe}SANS_DIPLOME_CEP"
        ],
        indicateurs_7c[total],
    )

    indicateurs_7c[
        f"PART_{groupe}_BAC_OU_PLUS_PCT"
    ] = pourcentage(
        indicateurs_7c[
            f"{groupe}_NON_SCOL_BAC_OU_PLUS"
        ],
        indicateurs_7c[total],
    )

    indicateurs_7c[
        f"PART_{groupe}_DIPLOME_SUP_PCT"
    ] = pourcentage(
        indicateurs_7c[
            f"{groupe}_NON_SCOL_DIPLOME_SUP"
        ],
        indicateurs_7c[total],
    )

    indicateurs_7c[
        f"PART_{groupe}_BAC5_PLUS_PCT"
    ] = pourcentage(
        indicateurs_7c[
            f"{prefixe}BAC5_PLUS"
        ],
        indicateurs_7c[total],
    )

In [18]:
ecarts_diplomes = {}

for groupe, informations in groupes_diplomes.items():
    total = informations["total"]
    prefixe = informations["prefixe"]

    composantes = [
        f"{prefixe}{code_diplome}"
        for code_diplome
        in modalites_diplomes
    ]

    ecart = (
        indicateurs_7c[total]
        - indicateurs_7c[
            composantes
        ].sum(axis=1)
    ).abs()

    ecarts_diplomes[groupe] = (
        ecart.max()
    )

for code_tranche in tranches_scolarite:
    population_reference = indicateurs_7c[
        f"POP_REFERENCE_SCOLARITE_{code_tranche}"
    ]

    population_scolarisee = indicateurs_7c[
        f"POP_SCOLARISEE_{code_tranche}"
    ]

    assert (
        population_scolarisee
        <= population_reference + 0.01
    ).all()

assert ecarts_diplomes["POP"] < 0.1
assert ecarts_diplomes["HOMMES"] < 0.1
assert ecarts_diplomes["FEMMES"] < 0.1
assert indicateurs_7c["CODGEO"].is_unique
assert len(indicateurs_7c) == len(profil_j6)

controle_7c = pd.DataFrame(
    {
        "CONTROLE": [
            "Nombre de communes",
            "Écart maximal diplômes population",
            "Écart maximal diplômes hommes",
            "Écart maximal diplômes femmes",
            "Présence de Paris 75056",
            "Présence de Saint-Denis 93066",
        ],
        "VALEUR": [
            len(indicateurs_7c),
            ecarts_diplomes["POP"],
            ecarts_diplomes["HOMMES"],
            ecarts_diplomes["FEMMES"],
            "75056" in indicateurs_7c["CODGEO"].values,
            "93066" in indicateurs_7c["CODGEO"].values,
        ],
    }
)

display(controle_7c)

print("Contrôles J7c validés ✅")

,CONTROLE,VALEUR
0,Nombre de communes,1266
1,Écart maximal diplômes population,0.00002
2,Écart maximal diplômes hommes,0.00002
3,Écart maximal diplômes femmes,0.00002
4,Présence de Paris 75056,True
5,Présence de Saint-Denis 93066,True


Contrôles J7c validés ✅


In [21]:
#Export
FICHIER_SORTIE_7C = (
    DOSSIER_INTERIM
    / "j7/j7c_scolarisation_diplomes_idf_2023.csv"
)

FICHIER_CONTROLE_7C = (
    DOSSIER_INTERIM
    / "j7/j7c_controle_scolarisation_diplomes_idf_2023.csv"
)

enregistrer_csv(
    indicateurs_7c,
    FICHIER_SORTIE_7C,
)

enregistrer_csv(
    controle_7c,
    FICHIER_CONTROLE_7C,
)

print("J7c terminé ✅")

Fichier CSV créé : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\interim\j7\j7c_scolarisation_diplomes_idf_2023.csv
Fichier CSV créé : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\interim\j7\j7c_controle_scolarisation_diplomes_idf_2023.csv
J7c terminé ✅
